## How to Combine GPT-oss with RAG 

The tutorial is developed based on openai-cookbook [example 1](https://github.com/openai/openai-cookbook/blob/main/examples/fine-tuned_qa/ft_retrieval_augmented_generation_qdrant.ipynb) and [example 2](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_combine_GPT4o_with_RAG_Outfit_Assistant.ipynb).

The aim of this notebook is to walk through a example of how to do Retrieval Augmented Generation (RAG). There are many great open source frameworks out there that can assist in the creation of RAG pipelines. In this notebook we choose to use the transformer library, standard Python libraries, and ultimately code our RAG from scratch so that you gain a fundamental understanding of how RAG works.

We are using [Design Safe user guide](https://designsafe-ci.org/user-guide/) as our data source.

We'll cover the following steps:

1. Data Preparation
2. RAG Prompt
   
Overall, the GPT-oss + RAG approach offers a fast, powerful, and flexible solution, leveraging the strengths of both generative and retrieval-based AI techniques.

#### Environment Setup

In [1]:
import json
import os
import time

import pandas as pd
from collections import defaultdict
import numpy as np
from tqdm import tqdm

import warnings

tqdm.pandas()
%matplotlib inline
%env HF_HOME={os.environ['SCRATCH']}/HFHOME

env: HF_HOME=/scratch/07980/sli4/HFHOME


### Data Preparation

We'll slice DesignSafe-CI's user documentation into chunks. Then generate embeddings for each chunk.

In [2]:
from pathlib import Path
from sentence_transformers import SentenceTransformer

def md_to_dataframe(directory_path, max_words=1024, overlap=128):
    """
    Recursively finds all .md files, reads them, and chunks the 
    content based on a maximum word length with an overlap.
    """
    data = []
    # Recursively find all .md files
    md_files = list(Path(directory_path).rglob('*.md'))
    
    for file_path in tqdm(md_files, desc="Parsing MD files"):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        words = content.split()
        for i in range(0, len(words), max_words - overlap):
            chunk_words = words[i : i + max_words]
            chunk = " ".join(chunk_words)
            if chunk.strip():
                data.append({
                    'content': chunk
                })
                    

    df = pd.DataFrame(data)
    return df

embedding_model = SentenceTransformer(
    'BAAI/bge-m3', 
    cache_folder=f"{os.environ['SCRATCH']}/data/training/"
)

def generate_embedding_from_dataframe(df: pd.DataFrame, text_column: str = "content"):
    """
    Generates embeddings for DataFrame.
    """
    batch_size = 64
    texts = df[text_column].tolist()
    
    pbar = tqdm(total=len(texts), desc="Generating embeddings")
    embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        batch_embeddings = embedding_model.encode(batch, batch_size=batch_size)
        embeddings.extend(batch_embeddings)
        pbar.update(len(batch))
        
    pbar.close()
    
    # Convert embeddings to list of lists
    embeddings_list = [embedding.tolist() for embedding in embeddings]
    
    # Create a temporary DataFrame to hold the embeddings
    temp_df = df.copy()
    temp_df["embeddings"] = embeddings_list
    temp_df["id"] = temp_df.index
    
    return temp_df


In [3]:
#!mkdir -p $SCRATCH/data/training
#!git clone https://github.com/DesignSafe-CI/DS-User-Guide.git $SCRATCH/data/training
md_directory = f"{os.environ['SCRATCH']}/data/training/user-guide"
df_markdown = md_to_dataframe(md_directory)
final_df = generate_embedding_from_dataframe(df_markdown, text_column="content")
print(final_df.head())

Generating embeddings: 100%|██████████| 268/268 [00:23<00:00, 11.51it/s]

                                             content  \
0  # Account Help ## Password Reset To **reset yo...   
1  # Analysis Tools & Applications {% include-mar...   
2  # DesignSafe Data Depot The <a href="https://w...   
3  !!! attention "This page has been deleted." Pl...   
4  # Metadata Dictionaries To understand the cont...   

                                          embeddings  id  
0  [-0.001927413628436625, 0.015702996402978897, ...   0  
1  [-0.038473911583423615, 0.001322947209700942, ...   1  
2  [-0.030034348368644714, -0.03599155321717262, ...   2  
3  [-0.010431998409330845, 0.01448755245655775, -...   3  
4  [-0.06541164964437485, -0.02324269898235798, -...   4  


### Building the Matching Algorithm

In this section, we'll develop a cosine similarity retrieval algorithm to find related content in our dataframe. We'll utilize our custom cosine similarity function for this purpose. 

Most standard databases come with their own search functions, which simplify the subsequent steps. However, we aim to demonstrate that the matching algorithm can be tailored to meet specific requirements, such as a particular threshold or a specified number of matches returned.

The`find_similar_questions` function accepts four parameters:

- `embedding`: The embedding for which we want to find a match.
- `embeddings`: A list of embeddings to search through for the best matches.
- `threshold` : This parameter specifies the minimum similarity score for a match to be considered valid. A higher threshold results in closer (better) matches, while a lower threshold allows for more items to be returned, though they may not be as closely matched to the initial embedding.
- `top_k`: This parameter determines the number of items to return that exceed the given threshold. These will be the top-scoring matches for the provided embedding.

In [4]:
def cosine_similarity_manual(vec1, vec2):
    """Calculate the cosine similarity between two vectors."""
    vec1 = np.array(vec1, dtype=float)
    vec2 = np.array(vec2, dtype=float)


    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)


def find_similar_chunks(input_embedding, embeddings, threshold=0, top_k=2):
    """Find the most similar chunks based on cosine similarity."""
    
    # Calculate cosine similarity between the input embedding and all other embeddings
    similarities = [(index, cosine_similarity_manual(input_embedding, vec)) for index, vec in enumerate(embeddings)]
    
    # Filter out any similarities below the threshold
    filtered_similarities = [(index, sim) for index, sim in similarities if sim >= threshold]
    
    # Sort the filtered similarities by similarity score
    sorted_indices = sorted(filtered_similarities, key=lambda x: x[1], reverse=True)[:top_k]

    # Return the top-k most similar items
    return sorted_indices


In [5]:
def find_matching_chunks_with_rag(query, data_with_embeddings):

   """Take the input query and find the most similar chunks based on cosine similarity."""
   
   # Select the embeddings from the DataFrame.
   embeddings = data_with_embeddings['embeddings'].tolist()
   
   similar_chunks = []
   # Generate the embedding for the input item
   input_embedding = batch_embeddings = embedding_model.encode(query)
   # Find the most similar items based on cosine similarity
   similar_indices = find_similar_chunks(input_embedding, embeddings, threshold=0)
   similar_chunks += [data_with_embeddings.iloc[i[0]] for i in similar_indices]
    
   return similar_chunks

Our main function get_prompt serves as the workhorse for generating prompts for RAG. It does this by retrieving chunks related to user query with our search function. 

In [6]:
def get_orig_prompt(query):    

    instruction = """Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"""
    
    prompt = [{"role": "system", "content": instruction}] + [
        {
            "role": "user",
            "content": f"""Question: {query}\n\n\n\nAnswer:"""
        },
    ]
    return prompt


In [7]:
def get_rag_prompt(query):

    # Query for similar chunks 
    ps = find_matching_chunks_with_rag(query, final_df)
    
    instruction = """Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"""
    
    Context = []
    for p in ps:
        Context.append(p["content"])
    
    rag_prompt = [{"role": "system", "content": instruction}] + [
    {
        "role": "user",
        "content": f"""Question: {query}\n\nContext: {Context}\n\nAnswer:"""
    },
    ]
    return rag_prompt



### Inference with RAG prompt

We can use gpt-oss for inference. 

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

# Load the model 
model_kwargs = dict(attn_implementation="eager", torch_dtype="auto", use_cache=True, device_map="auto")
model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs).cuda()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
def generate_answer(message):
    input_ids = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    gen_kwargs = {
        "max_new_tokens": 512, 
        "do_sample": True, 
        "temperature": 0.6, 
        "top_p": None, 
        "top_k": None
    }

    output_ids = model.generate(input_ids, **gen_kwargs)
    
    # Slice the output to exclude the original prompt tokens
    input_length = input_ids.shape[1]
    generated_ids = output_ids[0][input_length:]
    
    raw_response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    if "<|assistantfinal|>" in raw_response:
        final_answer = raw_response.split("<|assistantfinal|>")[-1]
    elif "assistantfinal" in raw_response: 
        final_answer = raw_response.split("assistantfinal")[-1]
    else:
        final_answer = raw_response 
        
    final_answer = final_answer.replace(tokenizer.eos_token, "").strip()
    return final_answer
    


In [10]:
query = "What is DesignSafe?"    
message=get_orig_prompt(query)
answer=generate_answer(message)
answer

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


"I don't know."

In [11]:
query = "What is DesignSafe?"   
message=get_rag_prompt(query)
message

[{'role': 'system',
  'content': "Answer the following Question based on the Context only. Only answer from the Context. If you don't know the answer, say 'I don't know'.\n\n"},
 {'role': 'user',
  'content': 'Question: What is DesignSafe?\n\nContext: ["# Cybersecurity Policy DesignSafe is an open CI that enables and supports leading-edge scientific discovery and promotes science and technology education. While it must be a widely accessible platform, a balanced approach to cybersecurity is required to support the confidentiality and integrity of the CI by architecting for security best practices, employing risk-based methodologies, and establishing a common project-wide approach to meet the needs of all NHERI awardees. DesignSafe implements cybersecurity based upon TACC\'s well-established approach that is compliant with the federal standards in the Federal Information Security Management Act (FISMA) in NIST Special Publication 800-53 Revision 4. We implement authentication, authoriza

In [14]:
answer=generate_answer(message)
answer

'DesignSafe is an open cyber‑infrastructure (CI) platform that enables and supports leading‑edge scientific discovery and promotes science and technology education. It provides researchers with resources for sharing, finding, analyzing, and publishing data; running numerical simulations and high‑performance computing; and integrating diverse datasets, particularly for natural‑hazard research.'

### Evaluating RAG with an LLM Judge

Now that we have successfully generated an answer using our RAG pipeline, we need a way to evaluate its quality. In traditional NLP workflows, static metrics like BLEU or ROUGE were the go-to. 

However, **traditional metrics fall dangerously short for free-form Question Answering**. BLEU and ROUGE rely on measuring exact word overlap (n-grams) between the generated answer and a pre-written reference answer. Because an LLM can generate a perfectly accurate and fluent answer using entirely different vocabulary or sentence structure than the reference, word-overlap metrics often unfairly penalize good answers or mistakenly reward fluent nonsense.

To solve this, utilizing an **"LLM-as-a-Judge"** has become the industry standard for automated, scalable evaluation during rapid development. 

For this tutorial, we will use a technique called **self-evaluation**. We will take the local model we used for generation and ask it to critique its own work based on a strict rubric.

#### Step 1: Defining the Rubric
The `get_judge_prompt` function constructs the grading instructions. Instead of asking a vague question like "Is this a good answer?", we instruct the model to evaluate two specific metrics on a 1-5 scale:

* **Context Relevance:** Evaluates the *retrieval* step. Did our cosine similarity search actually pull relevant documentation, or is it noise?
* **Faithfulness:** Evaluates the *generation* step. Did the model hallucinate outside information, or did it stick strictly to the retrieved context?

In [1]:
def get_judge_prompt(query, context, system_answer):
    instruction = """You are an impartial expert judge evaluating a Retrieval-Augmented Generation (RAG) system.
You will be provided with a User Question, the Retrieved Context, and the System's Answer.

Evaluate the system on two metrics, scoring each from 1 to 5:
1. Context Relevance: Does the Retrieved Context contain the necessary information to answer the User Question? (1 = completely irrelevant, 5 = perfectly relevant)
2. Faithfulness: Is the System's Answer completely derived from the Retrieved Context, without adding outside hallucinations? (1 = major hallucinations, 5 = perfectly grounded)

Provide a brief 1-sentence reasoning for each score, followed by the score itself."""

    prompt = [{"role": "system", "content": instruction}] + [
        {
            "role": "user",
            "content": f"Question: {query}\n\nContext: {context}\n\nSystem Answer: {system_answer}\n\nEvaluation:"
        }
    ]
    return prompt

**Step 2: Running the Evaluation**
To run the evaluation, our pipeline needs three pieces of data:

- The user's original Question.

- The exact Context retrieved by the vector search.

- The System Answer generated by the model.

We pass these three elements into our judge prompt, and then feed that prompt right back into our generate_answer function. The model will output a short critique of its own work, demonstrating how you can automate quality assurance in a RAG pipeline.

In [13]:
# 1. We need the same query and answer you just generated
query = "What is DesignSafe?"
# 'answer' variable is already stored in memory from your previous cell

# 2. Re-fetch the exact context that was used to generate that answer
retrieved_chunks = find_matching_chunks_with_rag(query, final_df)
context_used = [chunk["content"] for chunk in retrieved_chunks]

# 3. Build the prompt for the Judge LLM
judge_message = get_judge_prompt(query, context_used, answer)

# 4. Generate the evaluation using your existing model!
print("Generating evaluation... (this may take a moment)")
evaluation_result = generate_answer(judge_message)

print("\n=== LLM Judge Evaluation ===")
print(evaluation_result)

Generating evaluation... (this may take a moment)

=== LLM Judge Evaluation ===
Context Relevance: The provided context clearly describes DesignSafe as an open cyberinfrastructure platform for scientific discovery, so it is highly relevant to the question. 5  
Faithfulness: The system’s answer (“I don’t know”) adds no information from the context and fails to answer the question, thus it is not faithful. 1


### The Gold Standard: Human-in-the-Loop (HITL)

While an LLM Judge is a fantastic tool for quickly scaling evaluations and catching obvious errors during development, it is important to state that **Human-in-the-Loop evaluation remains the absolute gold standard.** Human domain experts are the only reliable way to establish true ground-truth accuracy, catch nuanced reasoning errors, and account for biases.

#### 🛠️ Try It Yourself: You Be The Judge
Now it's your turn to test both the RAG pipeline and the LLM Judge:
1. Come up with a new, specific question related to DesignSafe.
2. Run your question through the RAG generation cell.
3. Run the evaluation cell to see what the LLM Judge thinks.
4. Compare: Read the retrieved context and the generated answer yourself. Do you agree with the LLM's 1-5 scores? Did it miss a hallucination, or penalize a good answer unfairly?